# Active few-shot learning na przypadkach SAML-D — symulacja z analitykiem-wyrocznią

**Pytanie.** Czy model, który sam wybiera, który przypadek analityk ma oznaczyć jako następny, osiąga tę samą trafność przy mniejszej liczbie etykiet niż wybór losowy?

**Ustawienie (per fold, tylko typologie testowe foldu, niewidziane w treningu):**
- Przypadki typologii testowych dzielimy po grupach kont: **pula** (nieoznaczone, model może prosić o etykietę) i **zbiór oceny** (odłożony, nigdy nie oznaczany). Grupy są rozłączne, więc ten sam pracz nie trafia na obie strony.
- Start: po **1 oznaczonym przypadku na typologię** (4 etykiety, losowo).
- Każda runda: strategia wybiera 1 przypadek z puli, wyrocznia odsłania jego typologię, prototyp tej typologii się przelicza. Bez trenowania: nowa etykieta po prostu dołącza do prototypu.
- 20 rund (budżet 4 → 24 etykiety), 30 powtórzeń na fold, 5 foldów.
- Ocena: trafność zbalansowana (średni recall typologii) na zbiorze oceny po każdej rundzie.

**Strategie:**
- `random`: losowy przypadek z puli (punkt odniesienia).
- `margin`: przypadek z najmniejszą różnicą odległości do dwóch najbliższych prototypów (najbardziej niepewny).
- `typical`: w stylu TypiClust (Hacohen i in., ICML 2022). Pula razem z oznaczonymi jest dzielona na |oznaczone|+1 klastrów. Z największego klastra bez etykiety wybierany jest przypadek najbardziej typowy, czyli o najmniejszej średniej odległości do sąsiadów.
- `hybrid`: `typical` przez pierwsze 4 zapytania (rozgrzewka), potem `margin`, jak u Lorenza i in. (2020, AML).
- `kcenter`: przypadek najdalszy od wszystkich już oznaczonych (pokrycie przestrzeni). Tylko raportowany.

**Reprezentacje:** `raw`, czyli standaryzowane cechy przypadku (główna, najsilniejsza w kampanii), oraz `protonet`, czyli embedding modelu z kampanii z rejestru (raportowany, jeśli artefakt jest dostępny).

**Kryterium (ustalone przed uruchomieniem na SAML-D), reprezentacja `raw`:** Δ = trafność(`margin`) − trafność(`random`), uśredniona po budżetach 8–16 etykiet, liczona per fold.
- średnia Δ ≥ +0,03 **i** dolna granica 95% CI (t, n = 5 foldów) > 0 → **PASS**: active learning ma wartość, idzie do pipeline'u (komponenty `score` + `ingest`).
- w przeciwnym razie → **martwy dla `margin`**. `hybrid`, `typical` i `kcenter` są raportowane, ale nie ratują werdyktu, bo nie były kryterium.

**Rozważona i odrzucona zmiana kryterium:** literatura (TypiClust, Lorenz 2020) sugerowała, żeby kryterium główne przenieść na `hybrid`. Suchy bieg na danych syntetycznych tego nie poparł: `typical` przegrał tam z losowym, bo koncentrował etykiety na typologiach o wielu skupiskach i zaniedbywał inne. Zmiana kryterium bez dowodu byłaby kolejnym rozwidleniem, więc kryterium główne zostaje takie jak w pierwszej wersji (`margin`). Dodatkowe strategie są tylko raportowane.

**Znane ryzyko, zapisane z góry:** przy bardzo małym budżecie wybór „najbardziej niepewnych” bywa gorszy od losowego, bo łapie przypadki odstające i nie pokrywa klas (problem zimnego startu). Jeśli tak wyjdzie, to jest wynik.

Kernel: `fsl (py3.10)`. Czas (do sprawdzenia): kilka minut na CPU.

In [ ]:
import json, time
from pathlib import Path
from types import SimpleNamespace
import numpy as np, torch
from fsl.config import TrainConfig
from fsl.data.registry import get_loaders

HOME = Path("/home/jupyter")
PROJECT, BUCKET, DS = "dark-data-discovery", "dark-data-discovery-fsl-data", "samld-case"
RESULTS = HOME / "data_cache/samld_case_active_learning.json"
SEED_PER_CLASS, ROUNDS, REPS, POOL_SHARE = 1, 20, 30, 0.6
WARMUP = 4                                 # hybrid: tyle pierwszych zapytan wg typowosci, potem margin
BUDGETS = (8, 12, 16)                      # kryterium: srednia z tych budzetow

cfg0 = TrainConfig(project=PROJECT, bucket=BUCKET, dataset=DS, seed=0, fold=0,
                   n_way=4, k_shot=5, query=5, log_to_vertex=False)
load_frozen, build = get_loaders(cfg0)
bundle, sha = load_frozen(cfg0)
sch, W = bundle["schema"], bundle["windows"]
LAB = W[sch["label_col"]].to_numpy().astype(str)
GRP = W[sch["group_col"]].to_numpy()
print(f"samld-case sha {sha[:12]} | przypadkow {len(W):,} | foldow {sch['n_folds']}")

In [ ]:
def fold_matrix(k):
    # Standaryzacja tego foldu (statystyki z jego klas treningowych) - jak w pipelinie.
    c = TrainConfig(project=PROJECT, bucket=BUCKET, dataset=DS, seed=0, fold=k,
                    n_way=4, k_shot=5, query=5, log_to_vertex=False)
    _, _, te = build(bundle, c)
    return np.asarray(te.X, dtype=np.float32)


def load_protonet(k):
    # Najnowszy model kampanii dla foldu k z gs://<bucket>/models/samld-case/; None, jesli brak.
    try:
        import tempfile
        from google.cloud import storage
        from fsl.models.registry import get_encoder
        b = storage.Client(project=PROJECT).bucket(BUCKET)
        best = None
        for bl in b.list_blobs(prefix=f"models/{DS}/"):
            if bl.name.endswith("architecture.json"):
                a = json.loads(bl.download_as_text())
                if int(a.get("fold", -1)) == k and (best is None or bl.updated > best[0]):
                    best = (bl.updated, bl.name.rsplit("/", 1)[0], a)
        if best is None:
            return None
        d = Path(tempfile.mkdtemp())
        b.blob(f"{best[1]}/model.pt").download_to_filename(str(d / "model.pt"))
        a = best[2]
        m = get_encoder(SimpleNamespace(encoder=a.get("encoder", "mlp"), in_channels=a["in_channels"],
                                        embedding_hid=a["embedding_hid"], mlp_depth=a.get("mlp_depth", 2),
                                        mlp_dropout=a.get("mlp_dropout", 0.0)))
        m.load_state_dict(torch.load(d / "model.pt", map_location="cpu"))
        return m.eval()
    except Exception as e:                      # brak dostepu / artefaktu - reprezentacja pominieta
        print(f"  fold {k}: protonet niedostepny ({type(e).__name__}: {e})")
        return None


def typical_pick(Z, pool, L, cand, rng, knn=20):
    # TypiClust: |L|+1 klastrow na puli+oznaczonych; najwiekszy klaster bez etykiety;
    # w nim najbardziej typowy kandydat (najmniejsza srednia odleglosc do knn sasiadow).
    from sklearn.cluster import KMeans
    allp = np.union1d(pool, L)
    k = min(len(L) + 1, len(allp))
    km = KMeans(n_clusters=k, n_init=3, random_state=int(rng.integers(1 << 30))).fit(Z[allp])
    lab = dict(zip(allp.tolist(), km.labels_.tolist()))
    covered = {lab[i] for i in L.tolist()}
    sizes = np.bincount(km.labels_, minlength=k)
    order = [c for c in np.argsort(-sizes) if c not in covered]
    for c in order:
        members = np.array([i for i in cand if lab.get(int(i)) == c])
        if len(members):
            break
    else:
        members = cand
    if len(members) == 1:
        return members[0]
    D = np.sqrt(((Z[members][:, None] - Z[members][None]) ** 2).sum(-1))
    kk = min(knn, len(members) - 1)
    typ = 1.0 / (np.sort(D, axis=1)[:, 1:kk + 1].mean(1) + 1e-9)
    return members[np.argmax(typ)]


def balanced_acc(pred, y, n_cls):
    return float(np.mean([np.mean(pred[y == c] == c) for c in range(n_cls) if np.any(y == c)]))


def simulate(Z, y, grp, n_cls, strategy, rng):
    # Jedna krzywa uczenia: trafnosc zbalansowana po kazdej rundzie (budzet n_cls*SEED .. +ROUNDS).
    ug = np.unique(grp)
    pool_g = set(rng.choice(ug, int(round(POOL_SHARE * len(ug))), replace=False).tolist())
    in_pool = np.array([g in pool_g for g in grp])
    pool, ev = np.flatnonzero(in_pool), np.flatnonzero(~in_pool)
    if any(not np.any(y[pool] == c) or not np.any(y[ev] == c) for c in range(n_cls)):
        return None                                      # losowanie bez jakiejs klasy po ktorejs stronie
    labeled = [int(rng.choice(pool[y[pool] == c])) for c in range(n_cls) for _ in range(SEED_PER_CLASS)]
    curve = []
    for r in range(ROUNDS + 1):
        L = np.array(labeled)
        P = np.stack([Z[L[y[L] == c]].mean(0) for c in range(n_cls)])
        d_ev = ((Z[ev][:, None] - P[None]) ** 2).sum(-1)
        curve.append(balanced_acc(d_ev.argmin(1), y[ev], n_cls))
        if r == ROUNDS:
            break
        cand = np.setdiff1d(pool, L)
        if strategy == "random":
            pick = rng.choice(cand)
        elif strategy == "margin":
            dc = np.sort(np.sqrt(((Z[cand][:, None] - P[None]) ** 2).sum(-1)), axis=1)
            pick = cand[np.argmin(dc[:, 1] - dc[:, 0])]
        elif strategy == "kcenter":
            dmin = np.sqrt(((Z[cand][:, None] - Z[L][None]) ** 2).sum(-1)).min(1)
            pick = cand[np.argmax(dmin)]
        elif strategy == "typical" or (strategy == "hybrid" and r < WARMUP):
            pick = typical_pick(Z, pool, L, cand, rng)
        elif strategy == "hybrid":
            dc = np.sort(np.sqrt(((Z[cand][:, None] - P[None]) ** 2).sum(-1)), axis=1)
            pick = cand[np.argmin(dc[:, 1] - dc[:, 0])]
        labeled.append(int(pick))
    return curve


STRATS = ("random", "hybrid", "margin", "typical", "kcenter")


def run_fold(k, reps=REPS):
    test = bundle["folds"][k]["test"]
    rows = np.flatnonzero(np.isin(LAB, test))
    y = np.array([test.index(c) for c in LAB[rows]])
    reps_out = {}
    X = fold_matrix(k)[rows]
    reprs = {"raw": X}
    m = load_protonet(k)
    if m is not None:
        with torch.no_grad():
            reprs["protonet"] = m(torch.from_numpy(X)).numpy()
    for rn, Z in reprs.items():
        curves = {s: [] for s in STRATS}
        for rep in range(reps):
            for s in STRATS:                             # ten sam podzial pula/ocena i ten sam start dla strategii
                c = simulate(Z, y, GRP[rows], len(test), s, np.random.default_rng(10_000 * k + rep))
                if c is not None:
                    curves[s].append(c)
        reps_out[rn] = {s: np.array(v) for s, v in curves.items()}
    return reps_out

In [ ]:
t0 = time.time()
RES = {k: run_fold(k) for k in range(sch["n_folds"])}
print(f"symulacja: {time.time() - t0:.0f} s\n")

budget = lambda r: len(bundle["folds"][0]["test"]) * SEED_PER_CLASS + r
B_IDX = [b - budget(0) for b in BUDGETS]
for rn in ("raw", "protonet"):
    ks = [k for k in RES if rn in RES[k]]
    if not ks:
        continue
    print(f"=== reprezentacja: {rn} (foldy {ks}) ===")
    print("etykiety  " + "  ".join(f"{s:>8s}" for s in STRATS))
    for r in (0, 2, 4, 8, 12, 16, 20):
        vals = [np.mean([RES[k][rn][s][:, r].mean() for k in ks]) for s in STRATS]
        print(f"{budget(r):>8d}  " + "  ".join(f"{v:8.3f}" for v in vals))
    print()

try:
    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), sharey=True)
    for ax, rn in zip(axes, ("raw", "protonet")):
        ks = [k for k in RES if rn in RES[k]]
        for s, col in zip(STRATS, ("#888888", "#2a7d4f", "#C44E52", "#DD8452", "#4C72B0")):
            if ks:
                m = np.mean([RES[k][rn][s].mean(0) for k in ks], axis=0)
                ax.plot([budget(r) for r in range(len(m))], m, label=s, color=col, lw=2)
        ax.set_title(f"{rn}: trafnosc zbalansowana vs liczba etykiet")
        ax.set_xlabel("oznaczone przypadki (4 typologie testowe)")
        ax.grid(alpha=.3)
    axes[0].set_ylabel("trafnosc"); axes[0].legend()
    plt.tight_layout(); plt.show()
except ImportError:
    print("(brak matplotlib - tylko tabela)")

In [ ]:
from scipy import stats

def delta(rn, s, ks):
    return np.array([(RES[k][rn][s][:, B_IDX].mean() - RES[k][rn]["random"][:, B_IDX].mean()) for k in ks])

out = {"sha": sha, "budgets": BUDGETS, "reps": REPS, "rounds": ROUNDS, "pool_share": POOL_SHARE}
for rn in ("raw", "protonet"):
    ks = [k for k in RES if rn in RES[k]]
    if len(ks) < 2:
        continue
    for s in ("hybrid", "margin", "typical", "kcenter"):
        d = delta(rn, s, ks); t = stats.t.ppf(0.975, len(d) - 1); h = t * d.std(ddof=1) / np.sqrt(len(d))
        print(f"{rn:8s} {s:8s} vs random, budzety {BUDGETS}: per fold {np.round(d, 3)} | "
              f"srednia {d.mean():+.3f}, 95% CI [{d.mean() - h:+.3f}, {d.mean() + h:+.3f}]")
        out[f"{rn}-{s}"] = {"per_fold": d.tolist(), "mean": float(d.mean()), "ci_low": float(d.mean() - h),
                            "ci_high": float(d.mean() + h)}
    out[f"{rn}-curves"] = {s: np.mean([RES[k][rn][s].mean(0) for k in ks], axis=0).tolist() for s in STRATS}

p = out["raw-margin"]
verdict = ("PASS: active learning (margin) ma wartosc -> komponenty score + ingest w pipelinie"
           if p["mean"] >= 0.03 and p["ci_low"] > 0 else
           "MARTWY dla margin: wybor wg niepewnosci nie bije losowego w ustalonym kryterium")
out["verdict"] = verdict
print("\nWERDYKT:", verdict)

# ile etykiet potrzebuje margin, zeby dogonic random przy 24 etykietach (raw)
cur = out["raw-curves"]; target = cur["random"][-1]
for s_ in ("margin", "hybrid"):
    need = next((budget(r) for r, v in enumerate(cur[s_]) if v >= target), None)
    print(f"random przy {budget(ROUNDS)} etykietach: {target:.3f} | {s_} osiaga to przy: {need}")
    out[f"labels_to_match_random_max_{s_}"] = need
RESULTS.write_text(json.dumps(out, indent=2))
print("zapisano:", RESULTS)